# Indian E-Commerce Sales Analysis
## Notebook 02: Data Cleaning

**Goal:** Apply the cleaning plan from Notebook 01 and produce one clean, joined dataset for analysis.

Each step follows the same pattern: decision → fix → verification.

In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

In [2]:
sales = pd.read_csv("../data/raw/sales.csv")
customers = pd.read_csv("../data/raw/customers.csv")
products = pd.read_csv("../data/raw/products.csv")

rows_before = len(sales)
print("Sales rows before cleaning:", rows_before)

Sales rows before cleaning: 250000


## Step 1: Convert Dates

**Decision:** Date columns are stored as text. They must be real dates to group by month, calculate delivery time, and compare periods.

In [3]:
sales["Order_Date"] = pd.to_datetime(sales["Order_Date"], errors="coerce")
sales["Delivery_Date"] = pd.to_datetime(sales["Delivery_Date"], errors="coerce")

customers["Registration_Date"] = pd.to_datetime(customers["Registration_Date"], errors="coerce")
customers["Date_of_Birth"] = pd.to_datetime(customers["Date_of_Birth"], errors="coerce")

In [4]:
print(sales[["Order_Date", "Delivery_Date"]].dtypes)
print(customers[["Registration_Date", "Date_of_Birth"]].dtypes)

print("\nUnreadable dates:")
print("Order_Date:", sales["Order_Date"].isnull().sum())
print("Delivery_Date:", sales["Delivery_Date"].isnull().sum())
print("Registration_Date:", customers["Registration_Date"].isnull().sum())
print("Date_of_Birth:", customers["Date_of_Birth"].isnull().sum())

Order_Date       datetime64[us]
Delivery_Date    datetime64[us]
dtype: object
Registration_Date    datetime64[us]
Date_of_Birth        datetime64[us]
dtype: object

Unreadable dates:
Order_Date: 0
Delivery_Date: 0
Registration_Date: 0
Date_of_Birth: 0


In [5]:
early_delivery = sales["Delivery_Date"] < sales["Order_Date"]
print("Orders delivered BEFORE they were placed:", early_delivery.sum())

Orders delivered BEFORE they were placed: 0


**Result:** All date columns converted successfully (0 unreadable). 0 orders have a delivery date before the order date.

## Step 2: Fill Missing Coupon Codes

**Decision:** Notebook 01 proved every blank Coupon_Code has a discount of 0, so a blank means "no coupon used". Fill blanks with "No Coupon" instead of deleting 80% of the data.

In [6]:
sales["Coupon_Code"] = sales["Coupon_Code"].fillna("No Coupon")

In [7]:
print("Blank coupon codes left:", sales["Coupon_Code"].isnull().sum())
sales["Coupon_Code"].value_counts()

Blank coupon codes left: 0


Coupon_Code
No Coupon    199815
SAVE10        24997
DIWALI100     12609
FLAT50        12579
Name: count, dtype: int64

## Step 3: Keep Missing Ratings as Blank

**Decision:** Rating and Review_Text are blank for 129,970 orders. Notebook 01 showed that only delivered orders can be rated, and ~60% of delivered orders were rated. Filling blanks (for example with the average rating) would invent fake reviews, so they are left blank. Rating analysis will use rated, delivered orders only.

## Step 4: Fix Negative Total Amounts

**Decision:** 5 orders have a negative Total_Amount because flat coupons exceeded the order value plus shipping. A checkout never charges below ₹0, so these are capped at 0. A flag column keeps a record of which orders were changed.

In [8]:
sales["Negative_Total_Flag"] = sales["Total_Amount"] < 0
sales["Total_Amount"] = sales["Total_Amount"].clip(lower=0)

In [9]:
print("Minimum Total_Amount now:", sales["Total_Amount"].min())
print("Orders flagged:", sales["Negative_Total_Flag"].sum())

Minimum Total_Amount now: 0.0
Orders flagged: 5


## Step 5: Standardise State Names

**Decision:** All states are written in full except "UP". Rename it to "Uttar Pradesh" for consistent labels and so map charts can recognise it.

In [10]:
print(customers["State"].unique())

<ArrowStringArray>
[     'Punjab', 'West Bengal',     'Haryana',          'UP',   'Rajasthan',
   'Karnataka', 'Maharashtra',       'Delhi',  'Tamil Nadu',     'Gujarat']
Length: 10, dtype: str


In [11]:
sales["State"] = sales["State"].replace({"UP": "Uttar Pradesh"})
customers["State"] = customers["State"].replace({"UP": "Uttar Pradesh"})

In [12]:
print("'UP' left in sales:", (sales["State"] == "UP").sum())
print("'Uttar Pradesh' in sales:", (sales["State"] == "Uttar Pradesh").sum())
print(sorted(sales["State"].unique()))

'UP' left in sales: 0
'Uttar Pradesh' in sales: 32631
['Delhi', 'Gujarat', 'Haryana', 'Karnataka', 'Maharashtra', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Uttar Pradesh', 'West Bengal']


**Progress:** Steps 1–5 complete. Dates converted, coupon blanks filled, ratings intentionally kept blank, 5 negative totals capped and flagged, and state names standardised. No rows removed so far.

## Step 6: Create New Columns (Feature Engineering)

**Decision:** The analysis questions need columns that don't exist yet, such as year-month for trends, delivery time, and simple yes/no flags. These are created from existing columns.

In [13]:
sales["Year"] = sales["Order_Date"].dt.year
sales["Year_Month"] = sales["Order_Date"].dt.to_period("M").astype(str)
sales["Weekday"] = sales["Order_Date"].dt.day_name()

In [14]:
sales["Order_Time"].head()

0    08:20:00
1    20:05:00
2    14:59:00
3    14:33:00
4    07:19:00
Name: Order_Time, dtype: str

In [15]:
sales["Order_Hour"] = pd.to_datetime(sales["Order_Time"], format="%H:%M:%S", errors="coerce").dt.hour
print("Unreadable times:", sales["Order_Hour"].isnull().sum())

Unreadable times: 0


In [16]:
print(sales["Order_Hour"].min(), sales["Order_Hour"].max())

0 23


In [17]:
is_delivered = sales["Order_Status"] == "Delivered"
sales["Delivery_Days"] = (sales["Delivery_Date"] - sales["Order_Date"]).dt.days.where(is_delivered)

In [18]:
print("Orders with Delivery_Days:", sales["Delivery_Days"].notnull().sum())
sales["Delivery_Days"].describe()

Orders with Delivery_Days: 200139


count    200139.000000
mean          4.500892
std           1.710367
min           2.000000
25%           3.000000
50%           5.000000
75%           6.000000
max           7.000000
Name: Delivery_Days, dtype: float64

In [19]:
status_map = {
    "Delivered": "Completed",
    "Cancelled": "Lost",
    "Returned": "Lost",
    "Shipped": "Incomplete",
    "Processing": "Incomplete"
}
sales["Status_Group"] = sales["Order_Status"].map(status_map)

sales["Has_Coupon"] = sales["Coupon_Code"] != "No Coupon"
sales["Free_Shipping"] = sales["Shipping_Cost"] == 0

In [20]:
print(sales["Status_Group"].value_counts(), "\n")
print("Status_Group blanks:", sales["Status_Group"].isnull().sum())
print("Orders with coupon:", sales["Has_Coupon"].sum())
print("Orders with free shipping:", sales["Free_Shipping"].sum())

Status_Group
Completed     200139
Lost           25000
Incomplete     24861
Name: count, dtype: int64 

Status_Group blanks: 0
Orders with coupon: 50185
Orders with free shipping: 232459


## Step 7: Join Product and Customer Details

**Decision:** Sales has no Category, Brand, Gender or Customer_Tier. These are added from the products and customers tables using Product_ID and Customer_ID. Notebook 01 confirmed that the IDs are unique and there are no orphan records, so the join is safe.

In [21]:
product_cols = ["Product_ID", "Product_Name", "Category", "Brand", "Discount_Percent"]
customer_cols = ["Customer_ID", "Gender", "Customer_Tier", "Registration_Date"]

In [22]:
df = sales.merge(products[product_cols], on="Product_ID", how="left", validate="many_to_one")
df = df.merge(customers[customer_cols], on="Customer_ID", how="left", validate="many_to_one")

print("Rows after joining:", len(df))

Rows after joining: 250000


In [23]:
print("Missing after join:")
print(df[["Category", "Brand", "Gender", "Customer_Tier"]].isnull().sum())
df[["Order_ID", "Product_Name", "Category", "Brand", "Gender", "Customer_Tier"]].head()

Missing after join:
Category         0
Brand            0
Gender           0
Customer_Tier    0
dtype: int64


,Order_ID,Product_Name,Category,Brand,Gender,Customer_Tier
0,ORD0000000001,HP Laptop V7,Electronics,HP,male,Platinum
1,ORD0000000002,boAt Headphones V3,Electronics,boAt,male,Platinum
2,ORD0000000003,boAt Headphones V8,Electronics,boAt,female,Platinum
3,ORD0000000004,boAt Headphones V2,Electronics,boAt,female,Platinum
4,ORD0000000005,HP Laptop V10,Electronics,HP,male,Platinum


In [24]:
before_registration = df["Order_Date"] < df["Registration_Date"]
print("Orders placed before the customer registered:", before_registration.sum())

Orders placed before the customer registered: 0


**Result:** 0 orders were placed before the customer's registration date. Combined with the delivery-date check (0 deliveries before ordering), the dates are logically consistent across all three tables.

## Step 8: Final Checks and Save

In [25]:
remaining = df.isnull().sum()
remaining[remaining > 0]

Rating           129970
Review_Text      129970
Delivery_Days     49861
dtype: int64

In [26]:
print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(df))
print("Columns now:", df.shape[1])

Rows before cleaning: 250000
Rows after cleaning: 250000
Columns now: 37


In [27]:
df.to_csv("../data/processed/clean_sales.csv", index=False)
print("Saved!")

Saved!


## Cleaning Summary

| Step | Action | Result |
|---|---|---|
| 1 | Converted 4 date columns | 0 unreadable; 0 deliveries before order date |
| 2 | Filled Coupon_Code blanks with "No Coupon" | 199,815 filled |
| 3 | Kept Rating/Review_Text blank | Only delivered orders can be rated |
| 4 | Capped negative Total_Amount at 0 | 5 orders, flagged in Negative_Total_Flag |
| 5 | Renamed "UP" to "Uttar Pradesh" | In both sales and customers |
| 6 | Created new columns | Year, Year_Month, Weekday, Order_Hour, Delivery_Days, Status_Group, Has_Coupon, Free_Shipping |
| 7 | Joined product and customer details | Category, Brand, Gender, Customer_Tier and more; rows unchanged |
| 8 | Saved clean data | data/processed/clean_sales.csv |

**Rows before: 250,000. Rows after: 250,000. No orders were removed.**
Remaining blanks are intentional: Rating, Review_Text (not rated) and Delivery_Days (not delivered).

In [28]:
print(df["Gender"].value_counts(), "\n")
print(df["Customer_Tier"].value_counts())

Gender
male      125226
female    124774
Name: count, dtype: int64 

Customer_Tier
Platinum    187576
Gold         37065
Silver       25359
Name: count, dtype: int64
